In [68]:
import sys

sys.path.append("..")

import matplotlib.pyplot as plt
import numpy as np
import random
from pulp import *

In [69]:
from voting import (
    Candidate,
    Voter,
    Election,
    PluralityRule,
    BordaRule,
    VetoRule,
)

from voting.plotting import (
    plot_result,
    random_2d_point,
)


In [70]:
N_VOTERS = 8000
N_CANDIDATES = 2000

# strategies = [PluralityRule(), BordaRule(), VetoRule()]
strategies = [PluralityRule(), BordaRule()]

# Resample the election until every rule has a unique (non-tied) winner.
# A tie means margin 0 -> the LP cannot enforce a strict win, so we reject such
# cases up front. This guarantees a witness exists and the LP below returns Optimal.
while True:
    candidates = [
        Candidate(id=id, position=random_2d_point()) for id in range(N_CANDIDATES)
    ]
    voters = [Voter(position=random_2d_point()) for _ in range(N_VOTERS)]
    election = Election(candidates=candidates, voters=voters)
    election_result = election.compare(strategies)
    if not election_result.has_tie():
        break

# plot_result(election_result)

In [71]:
winners = election_result.winner_indices()
winners

{'plurality': 1404, 'borda': 380}

In [72]:
def add_gaussian_noise_2d(position: np.ndarray, cov_matrix, bounds = (-10, 10)):
    # cov_matrix is a 2x2 covariance matrix, e.g. [[sigma_x**2, 0], [0, sigma_y**2]]
    noisy_point = np.random.multivariate_normal(position, cov_matrix)
    noisy_point = np.clip(noisy_point, bounds[0], bounds[1])
    return noisy_point

cov = np.array([[0.25, 0], [0, 0.25]])

In [73]:
voters_history = []

In [74]:
while True:
    noisy_voters = [
        Voter(position=add_gaussian_noise_2d(voter.position, cov))
        for voter in voters
    ]

    noisy_election = Election(candidates=candidates, voters=noisy_voters)
    noisy_election_result = noisy_election.compare(strategies)

    noisy_winners = noisy_election_result.winner_indices()
    voters_history.append(noisy_voters)

    if noisy_winners != winners:
        break

    if len(voters_history) % 100 == 0:
        print(len(voters_history))

In [75]:
noisy_election = Election(candidates=candidates, voters=voters_history[-1])
noisy_election_result = noisy_election.compare(strategies)

# plot_result(noisy_election_result)

In [76]:
print(f'Voters history length: {len(voters_history)}')

winners = election_result.winner_indices()
noisy_winners = noisy_election_result.winner_indices()

print(f'Winners in primary election: {winners}')
print(f'Winenrs in noisy election: {noisy_winners}')

Voters history length: 1
Winners in primary election: {'plurality': 1404, 'borda': 380}
Winenrs in noisy election: {'plurality': 1697, 'borda': 380}
